# Tiered现金层拆分
0拟合、0新回测。按现有控制器顺序对3个已对账原生账户拆分现金权重，逐日检查最后现金权重与实际持仓相等。不把归因项当成删去风控后的收益。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tiered_cash_layers_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.tiered_cash_layers import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'tiered_cash_layers_20261014',
 'plan_sha256': 'a86a8bd6c630fb549168231b8a8a92f237725151d2c1ed2bbf2d5154b5675ba0',
 'decision': 'diagnostic_only_no_promotion',
 'objective_achieved': False,
 'new_fits': 0,
 'new_native_cases': 0,
 'reused_native_cases': 3,
 'sessions_per_cost': 944,
 'maximum_cash_identity_error': 0.0,
 'recorder_id': '9c1f9f1cf7b845c8a49d643572203702',
 'experiment_id': '522840418719482073',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/tiered_cash_layers_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/tiered_cash_layers_20261014/qlib_recorder/522840418719482073/9c1f9f1cf7b845c8a49d643572203702',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013',
 'region': 'us',
 'as_of': '2026-10-09',
 'qlib_docs': 'https://qlib.readthedocs.io/en/latest/component/recorder.html',
 'units': 'mean cash weight percentage points; binding columns are session counts',
 'scope': 'ord

# Tiered现金来源：主要来自基础分配输出

复用3个Qlib原生账户，每本944个交易区间，2023现金起点至2026-10-09来源快照。0拟合、0新增回测；初始10万美元，IBKR Pro Tiered与0/2/5bp各自原持仓路径。生产模型与release未修改。

先读取已冻结的未平滑预测和EMA/HOLD后目标，后者逐日与当前账本输入现金目标核对一致；再逐日重建实际净值、高水位、历史收益、控制器状态、目标和执行决定。投影结果、beta上限和状态均与原代码/原账本核对，按实际订单后的持仓计算现金权重。

|现金来源（平均权重百分点）|0bp|2bp|5bp|
|---|---:|---:|---:|
|未平滑模型输出|40.7875|40.7875|40.7875|
|上游EMA/持仓过滤净变化|-0.0370|-0.0370|-0.0370|
|现金底线/SOXL上限合法投影|-0.0000|-0.0000|-0.0000|
|停止周期投影增量|5.5703|6.0109|10.7859|
|近期亏损限制增量|0.5426|0.5438|0.3533|
|紧急回撤限制增量|1.2032|0.8707|6.2183|
|当日开盘HOLD增量|0.1621|0.1864|0.2364|
|已知价至成交价漂移|0.0009|0.0018|0.0009|
|整数股成交/费用归一化|-0.0018|-0.0077|-0.0126|
|实际开盘成交后现金|48.2277|48.3565|58.3327|

这些是权重百分点，不是收益百分点。各项按代码执行顺序作差，可精确加总，但不是独立因果效应，也不代表删掉对应规则后的收益。`raw_cash`是进入Stage1的已平滑上游目标，`model_cash`才是EMA前模型输出，二者不能混称。

2bp中40.79%的模型现金目标占最终48.36%平均现金约84%；风控三项合计增加7.43个百分点。源平滑净影响-0.037个百分点，HOLD约+0.186个百分点，取整/费用归一化约-0.008个百分点。现金底线和SOXL上限的这一层投影没有额外提高现金；这不表示基础模型自身的输出约束没有作用，也不能说明平滑对收益或风险影响很小。

5bp中停止周期增量10.79个百分点、紧急限制6.22个百分点，较2bp明显增大。beta进一步受到周期/近期亏损/紧急限制的日数分别为：2bp 246/46/32，5bp 411/34/189。此为有序的新增约束计数，某上限被前面更严格上限遮盖时不重复记为新增作用；不是各触发条件总次数。

未平滑模型现金目标按trade_date年度均值：2023约37.22%、2024约49.98%、2025约41.30%、2026约32.73%。这些目标跨费用情景相同，后续实际风控与持仓路径不同。年度分组使用trade_date，与某些旧收益报告按signal_date分组的边界会差一个会话。

决定：**diagnostic_only_no_promotion**。最大逐日现金分解误差为0（当前浮点输出）；独立的顺序约束测试通过。原生账户证据复用，不把重新分解当作新增收益验证。

本结果排除了“主要是平滑、HOLD或取整把现金抬高”的解释，但没有证明模型持有现金不合理。此前删除风控、现金回收、动态上限与重新武装对照均未取得稳定超额，不能从本表推出放宽约束。稳定净SOXX超额目标未完成。

下一步应检查基础分配器的现金判断及其训练目标，先核对已有同结构/同输入对照，避免再次搜索已失败的风控规则；需要的是能在独立验证中区分应当持有现金和应当参与市场的信息，而不是把40.79%现金机械改成股票。

风险：历史已复用，当前费率是投影，真实成交/分红现金流资格继承来源；本卡不产生可部署的新策略。完整来源对象、原始逐日分解在NAS Recorder中；Git仅保留本notebook、[汇总表](cash_layer_summary.csv)、plan与验证清单。
